In [ ]:
# tox21_bilstm.py  ──────────────────────────────────────────────────────────
import os, numpy as np, pandas as pd, tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, accuracy_score,
                             f1_score, precision_score, recall_score)
# 1. Load the data ---------------------------------------------------------------
df = pd.read_csv("../data/tox21.csv")                     
targets = ['NR-AR','NR-AR-LBD','NR-AhR','NR-Aromatase','NR-ER',
           'NR-ER-LBD','NR-PPAR-gamma','SR-ARE','SR-ATAD5',
           'SR-HSE','SR-MMP','SR-p53']
smiles = df["smiles"].astype(str).tolist()
y      = df[targets]
mask   = y.isna()                 # True=loss data
y_filled = y.fillna(0).values.astype("float32")


# 2. Character tokeniser -------------------------------------------------------
tok = tf.keras.preprocessing.text.Tokenizer(char_level=True)
tok.fit_on_texts(smiles)
sequences = tok.texts_to_sequences(smiles)
MAXLEN = 150                               # Cover 95% length of SMILES
X = tf.keras.preprocessing.sequence.pad_sequences(
        sequences, maxlen=MAXLEN, padding="post", truncating="post")

# 3. Split data --------------------------------------------------------------
X_tr, X_tmp, y_tr, y_tmp, m_tr, m_tmp = train_test_split(
        X, y_filled, mask.values, test_size=0.30, random_state=42)
X_va, X_te, y_va, y_te, m_va, m_te = train_test_split(
        X_tmp, y_tmp, m_tmp, test_size=0.50, random_state=42)

# 4. Build BiLSTM ------------------------------------------------------
def build_bilstm(vocab_size, emb_dim=128, hid=32, dropout=0.3, lr=1e-3):
    inp = tf.keras.Input(shape=(MAXLEN,), dtype="int32")
    x   = tf.keras.layers.Embedding(vocab_size, emb_dim, mask_zero=True)(inp)
    x   = tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(hid, dropout=dropout))(x)
    out = tf.keras.layers.Dense(len(targets), activation="sigmoid")(x)
    model = tf.keras.Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy")
    return model

model = build_bilstm(len(tok.word_index)+1,
                     emb_dim=128, hid=32, dropout=0.3, lr=1e-3)

# 5. Trainning the model（mask missing label） --------------------------------------------------
class MaskedBCE(tf.keras.losses.Loss):
    def call(self, y_true, y_pred):
        mask = tf.cast(tf.not_equal(y_true, -1.0), tf.float32)
        loss = tf.keras.backend.binary_crossentropy(y_true, y_pred)
        return tf.reduce_sum(loss * mask) / tf.reduce_sum(mask)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss=MaskedBCE())

early = tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)
model.fit(X_tr, y_tr, epochs=5, batch_size=128,
          validation_data=(X_va, y_va), callbacks=[early], verbose=2)

# 6. Evaluation index --------------------------------------------------------------
y_prob = model.predict(X_te, batch_size=256)
metrics = []
for i, col in enumerate(targets):
    valid = ~m_te[:, i]
    if valid.sum() == 0: continue
    yt, yp = y_te[valid, i], y_prob[valid, i]
    y_bin  = (yp >= 0.5).astype(int)
    auc = roc_auc_score(yt, yp) if len(np.unique(yt))==2 else np.nan
    acc = accuracy_score(yt, y_bin)
    f1  = f1_score(yt, y_bin, zero_division=0)
    pre = precision_score(yt, y_bin, zero_division=0)
    rec = recall_score(yt, y_bin, zero_division=0)
    metrics.append([col, auc, acc, f1, pre, rec])

df_metrics = pd.DataFrame(metrics,
             columns=["Target","AUC","Accuracy","F1","Precision","Recall"])
# df_metrics.to_csv("bilstm_tox21_metrics.csv", index=False)
print(df_metrics)

Epoch 1/5
44/44 - 17s - 394ms/step - loss: 0.3375 - val_loss: 0.2284
Epoch 2/5
44/44 - 11s - 248ms/step - loss: 0.2230 - val_loss: 0.2267
Epoch 3/5
44/44 - 11s - 247ms/step - loss: 0.2223 - val_loss: 0.2262
Epoch 4/5
44/44 - 10s - 222ms/step - loss: 0.2218 - val_loss: 0.2256
Epoch 5/5
44/44 - 15s - 347ms/step - loss: 0.2206 - val_loss: 0.2243
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 278ms/step
           Target       AUC  Accuracy   F1  Precision  Recall
0           NR-AR  0.765698  0.962766  0.0        0.0     0.0
1       NR-AR-LBD  0.673648  0.971347  0.0        0.0     0.0
2          NR-AhR  0.706391  0.887786  0.0        0.0     0.0
3    NR-Aromatase  0.564266  0.952968  0.0        0.0     0.0
4           NR-ER  0.587119  0.868085  0.0        0.0     0.0
5       NR-ER-LBD  0.492214  0.944496  0.0        0.0     0.0
6   NR-PPAR-gamma  0.513751  0.972946  0.0        0.0     0.0
7          SR-ARE  0.594675  0.854444  0.0        0.0     0.0
8        SR-ATAD5  0.572968  0.966148  0.0        0.0     

Improve with weights

In [ ]:
# tox21_bilstm_weighted
# ──────────────────────────────────────────────────────────────────────────
import os, random
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, accuracy_score, f1_score,
                             precision_score, recall_score)
from tqdm import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# 1. Load the data ────────────────────────────────────────────────────────────
CSV_PATH = "../data/tox21.csv"                   
df = pd.read_csv(CSV_PATH)

targets = ['NR-AR','NR-AR-LBD','NR-AhR','NR-Aromatase','NR-ER',
           'NR-ER-LBD','NR-PPAR-gamma','SR-ARE','SR-ATAD5',
           'SR-HSE','SR-MMP','SR-p53']

# label use -1 as sentinel；for mask loss data later
y_sen = df[targets].fillna(-1).astype('float32').values
smiles = df["smiles"].astype(str).tolist()

# 2. Character Tokeniser & PAD ──────────────────────────────────────────────
tok = tf.keras.preprocessing.text.Tokenizer(char_level=True)
tok.fit_on_texts(smiles)
seqs = tok.texts_to_sequences(smiles)

MAXLEN = 150
X = tf.keras.preprocessing.sequence.pad_sequences(
        seqs, maxlen=MAXLEN, padding="post", truncating="post")

# 3. Split the data ───────────────────────────────────────────────────
X_tr, X_tmp, y_tr, y_tmp = train_test_split(
        X, y_sen, test_size=0.30, random_state=SEED)
X_va, X_te, y_va, y_te = train_test_split(
        X_tmp, y_tmp, test_size=0.50, random_state=SEED)

# 4. Calculate pos_weight  (only use training set) ────────────────────────────────────
pos = (y_tr == 1).sum(axis=0).astype('float32')
neg = (y_tr == 0).sum(axis=0).astype('float32')
eps = 1e-6
pos_weight = (neg / (pos + eps))           # shape (12,)

# 5. Build Bi-LSTM ───────────────────────────────────────────────────
VOCAB = len(tok.word_index)+1
def build_model(vocab_size, emb_dim=128, hid=32, dropout=0.3):
    inp = tf.keras.Input(shape=(MAXLEN,), dtype="int32")
    x   = tf.keras.layers.Embedding(vocab_size, emb_dim,
                                    mask_zero=True)(inp)
    x   = tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(hid, dropout=dropout))(x)
    out = tf.keras.layers.Dense(len(targets), activation="sigmoid")(x)
    return tf.keras.Model(inp, out)

model = build_model(VOCAB)

# 6. Define Masked + Weighted BCE ────────────────────────────────────────
pos_w_const = tf.constant(pos_weight, dtype=tf.float32)

def masked_weighted_bce(y_true, y_pred):
    """
    y_true:  -1 = missing, 0/1 = valid label
    """
    valid = tf.not_equal(y_true, -1.0)                 # bool mask
    y_clean = tf.where(valid, y_true, 0.)              # occupied 0
    # element level BCE
    l = tf.keras.backend.binary_crossentropy(y_clean, y_pred)
    # Add category weight: Only applies to positive samples (y_clean==1)
    weights = tf.where(tf.equal(y_clean, 1.0),
                       pos_w_const, tf.ones_like(pos_w_const))
    l = l * weights
    # Take the average only at the valid position
    l = tf.reduce_sum(tf.where(valid, l, 0.)) \
        / tf.reduce_sum(tf.cast(valid, tf.float32))
    return l

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss=masked_weighted_bce)

# 7. Trainning the model ────────────────────────────────────────────────────────────────
early = tf.keras.callbacks.EarlyStopping(patience=3,
                                         restore_best_weights=True)
model.fit(X_tr, y_tr, epochs=10, batch_size=128,
          validation_data=(X_va, y_va), callbacks=[early], verbose=2)

# 8. Evaluation index ────────────────────────────────────────────────────────────
y_prob = model.predict(X_te, batch_size=256, verbose=0)

results = []
for i, col in enumerate(targets):
    valid = y_te[:, i] != -1
    if valid.sum() == 0:
        continue
    yt, yp = y_te[valid, i], y_prob[valid, i]
    y_bin  = (yp >= 0.5).astype(int)

    auc = roc_auc_score(yt, yp) if len(np.unique(yt)) == 2 else np.nan
    acc = accuracy_score(yt, y_bin)
    f1  = f1_score(yt, y_bin, zero_division=0)
    pre = precision_score(yt, y_bin, zero_division=0)
    rec = recall_score(yt, y_bin, zero_division=0)
    results.append([col, auc, acc, f1, pre, rec])
    

df_metrics = pd.DataFrame(results,
        columns=["Target","AUC","Accuracy","F1","Precision","Recall"])
stats_cols = ["AUC","Accuracy","F1","Precision","Recall"]
mean_row   = df_metrics[stats_cols].mean(skipna=True).to_frame().T
median_row = df_metrics[stats_cols].median(skipna=True).to_frame().T

mean_row.insert(0, "Target", "Overall Mean")
median_row.insert(0, "Target", "Overall Median")

df_metrics = pd.concat([df_metrics, mean_row, median_row], ignore_index=True)
# df_metrics.to_csv("BiLSTM_Tox21_metrics_weighted.csv", index=False)
print("\n=== Test-set metrics ===")
# print(df_metrics.to_string(index=False))
print(df_metrics)

Epoch 1/10
44/44 - 27s - 622ms/step - loss: 1.2699 - val_loss: 1.2560
Epoch 2/10
44/44 - 13s - 306ms/step - loss: 1.2116 - val_loss: 1.2256
Epoch 3/10
44/44 - 13s - 286ms/step - loss: 1.1662 - val_loss: 1.1658
Epoch 4/10
44/44 - 11s - 261ms/step - loss: 1.1242 - val_loss: 1.1597
Epoch 5/10
44/44 - 10s - 227ms/step - loss: 1.1036 - val_loss: 1.1462
Epoch 6/10
44/44 - 9s - 204ms/step - loss: 1.1012 - val_loss: 1.1326
Epoch 7/10
44/44 - 11s - 261ms/step - loss: 1.0843 - val_loss: 1.1426
Epoch 8/10
44/44 - 13s - 302ms/step - loss: 1.0949 - val_loss: 1.1128
Epoch 9/10
44/44 - 16s - 352ms/step - loss: 1.0913 - val_loss: 1.1326
Epoch 10/10
44/44 - 14s - 319ms/step - loss: 1.0920 - val_loss: 1.1421

=== Test-set metrics ===
            Target       AUC  Accuracy        F1  Precision    Recall
0            NR-AR  0.790932  0.932624  0.387097   0.292683  0.571429
1        NR-AR-LBD  0.771321  0.932187  0.323810   0.226667  0.566667
2           NR-AhR  0.784760  0.698113  0.355932   0.233983  0.7